# Citrus v4 — 60k target, 20% synthesized knowledge

This notebook **generates draft candidates**, not a trained model or an approved 60,000-record release. The repository has a 100-record reviewed pilot. Target: 12,000 explanatory/synthesis + 48,000 practical records across 14 categories.

Use a CUDA GPU and enable Internet. The teacher is an open-weight local model; no paid inference API or embedded credential is used. Begin with 200 new drafts because the prior unguided generation produced nonexistent Roblox APIs. Review actual output before raising the per-run limit.


In [ ]:
%pip install -q "transformers>=4.51,<5" "accelerate>=1,<2" "bitsandbytes>=0.45,<1" sentencepiece


In [ ]:
import torch
assert torch.cuda.is_available(), "Select a CUDA GPU before generation; no run started."
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import json, pathlib, urllib.request, hashlib
REPO = "gentle58715-ship-it/Citrus-Training-"
BRANCH = "dataset/citrus-60k-v4"
ROOT = pathlib.Path("/kaggle/working/citrus-v4-source")
ROOT.mkdir(parents=True, exist_ok=True)
# Resolve once, then download all inputs from that immutable commit.
with urllib.request.urlopen(f"https://api.github.com/repos/{REPO}/git/ref/heads/{BRANCH}", timeout=60) as response:
    REVISION = json.load(response)["object"]["sha"]
BASE = f"https://raw.githubusercontent.com/{REPO}/{REVISION}"
FILES = ["scripts/citrus_v4.py", "datasets/v4/config.json", "datasets/v4/sources.json",
         "datasets/v4/tool_schemas.json", "datasets/v4/reviews.json",
         "datasets/v4/candidates/authored_0001.jsonl",
         "datasets/v4/candidates/tool_replay_0001.jsonl"]
for name in FILES:
    target = ROOT / name
    target.parent.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(f"{BASE}/{name}", timeout=60) as response:
        target.write_bytes(response.read())
print("Pinned source revision:", REVISION)


In [ ]:
import sqlite3
OUT = pathlib.Path("/kaggle/working/citrus-v4-progress")
OUT.mkdir(parents=True, exist_ok=True)
RESUME_DB = ""  # Exact path to a previously attached progress.sqlite, if resuming.
if RESUME_DB:
    if (OUT / "progress.sqlite").exists():
        raise RuntimeError("Refusing to overwrite an existing checkpoint")
    source = sqlite3.connect(f"file:{RESUME_DB}?mode=ro", uri=True)
    destination = sqlite3.connect(OUT / "progress.sqlite")
    try:
        source.backup(destination)
    finally:
        destination.close()
        source.close()


In [ ]:
import subprocess, shutil
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"  # Teacher only; not the selected Citrus training model.
NEW_CANDIDATE_LIMIT = 200
MAX_MINUTES = 60
command = ["python", "-u", str(ROOT / "scripts/citrus_v4.py"), "generate",
           "--root", str(ROOT / "datasets/v4"), "--out", str(OUT), "--model", MODEL_ID,
           "--limit", str(NEW_CANDIDATE_LIMIT), "--minutes", str(MAX_MINUTES)]
try:
    subprocess.run(command, check=True)
finally:
    # Includes committed SQLite progress even if generation fails or is interrupted.
    archive = shutil.make_archive("/kaggle/working/Citrus-v4-progress", "zip", OUT)
    print("Checkpoint archive:", archive)


In [ ]:
status_path = OUT / "status.json"
if status_path.exists():
    print(status_path.read_text())
else:
    print("No completed generation checkpoint was exported; inspect the prior error and saved DB.")


## Review and continue

Save the output checkpoint before ending a session. New local-model output remains `draft` and is separate from the 100 authored pilot records. Import and review candidates without duplicating IDs, check factual accuracy and semantic diversity, and execute code in the appropriate environment. Tool trajectories must be produced by a real isolated executor, not invented by the model.

The full export command is blocked until 60,000 reviewed records match the category and 20% knowledge quotas. JSON validity, successful GPU execution and the requested count are not proof of dataset quality. No automatic GitHub write credential is embedded in this notebook.
